# Data Cleaning — SBSA DS Case Study 2027

All cleaning below is applied to **`SBSA DS Case Study 2027 copy.csv`** only. The original
`SBSA DS Case Study 2027.csv` is never read for writing and never modified.

Structure of this notebook: each cleaning step is one code cell, followed by a markdown cell
explaining **what was done** and **why**, so every decision can be quoted directly in the report.
The issues being fixed were identified in the audit documented in `anomaly.ipynb`.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('SBSA DS Case Study 2027 copy.csv')
print('Starting shape:', df.shape)

Starting shape: (10000, 36)


**What:** Loaded the working copy of the dataset (10,000 rows × 36 columns).

**Why:** All cleaning happens on the copy so the raw original stays intact as an audit trail —
if any cleaning decision is later questioned, we can always re-derive everything from the untouched source.

In [2]:
before = len(df)
df = df.drop_duplicates()                                    # 13 exact full-row copies
df = df.drop_duplicates(subset='CustomerID', keep='first')   # 14 IDs that appear twice with conflicting data
print(f'Removed {before - len(df)} duplicate rows. Rows remaining: {len(df)}')
print('CustomerID now unique:', df['CustomerID'].is_unique)

Removed 27 duplicate rows. Rows remaining: 9973
CustomerID now unique: True


**What:** Removed the 27 duplicated `CustomerID`s (27 surplus rows deleted, one row kept per ID).
- 13 were *exact* full-row copies — dropping them is risk-free, the information is identical.
- 14 IDs appeared twice with *conflicting* attribute values — we kept the first occurrence, a standard
  convention when there is no timestamp or other evidence to say which version is correct.

**Why:** Duplicated clients would be double-counted in every statistic and would skew averages,
segment sizes, and any model trained on the data (the same person effectively gets twice the weight).
`CustomerID` must be a unique key for joins and per-client feature engineering to work correctly.

In [3]:
minors = df[df['Age'] < 18]
lending = ['Credit_Card', 'Personal_Loan', 'Home_Loan', 'Vehicle_Loan']

issues = pd.DataFrame({
    'negative_age':        minors['Age'] < 0,
    'has_lending_product': minors[lending].any(axis=1),
    'married_or_divorced': minors['Marital_Status'].isin(['Married', 'Divorced', 'Widowed']),
    'tertiary_education':  minors['Education'].isin(["Bachelor's", 'Postgraduate', 'Diploma/Certificate']),
    'adult_employment':    minors['Employment_Type'].isin(['Full-time employed', 'Self-employed', 'Retired']),
    'tenure_exceeds_age':  minors['Customer_Tenure'] > minors['Age'],
})

print('Under-18 clients found:', len(minors))
print('\nBreakdown of implausibilities among them:')
print(issues.sum().to_string())
print('\nUnique under-18 records with at least one issue:', issues.any(axis=1).sum())

df = df[df['Age'] >= 18]
print('\nRows remaining after removal:', len(df))

Under-18 clients found: 107

Breakdown of implausibilities among them:
negative_age            3
has_lending_product    78
married_or_divorced    38
tertiary_education     40
adult_employment       56
tenure_exceeds_age     10

Unique under-18 records with at least one issue: 97

Rows remaining after removal: 9866


**What:** Removed every client with `Age < 18` — this includes the 3 impossible negative ages
(-10, -5, -1). The cell above also prints the number of **unique under-18 records that had at least one
data issue** (lending products, married/divorced status, tertiary education, adult employment,
tenure longer than their age, or a negative age).

**Why:**
1. **Business decision:** the solution will not target clients under 18, so they are out of scope regardless.
2. **Data quality:** an overwhelming share of these records are internally implausible — minors holding
   home loans, recorded as married, retired, or with postgraduate degrees. That pattern suggests these rows
   are corrupted (e.g. an age field error) rather than genuine children, so imputing or "fixing" them would
   be guesswork. Deleting a small, clearly unreliable slice (~1% of data) is safer than keeping noise.

In [4]:
print('Missing credit scores:', df['Credit_Score'].isna().sum())
print('Credit scores outside the standard 300-850 range:',
      ((df['Credit_Score'] < 300) | (df['Credit_Score'] > 850)).sum())
# Deliberate decision: NO imputation — missing scores stay as NaN.

Missing credit scores: 362
Credit scores outside the standard 300-850 range: 1066


**What:** Nothing was changed — the missing `Credit_Score` values are deliberately **left as NaN**.

**Why:** A credit score is a high-stakes attribute; inventing one via imputation could materially
misrepresent a client's risk. Leaving it blank keeps the rest of the row usable (demographics, products,
digital behaviour) for analyses that don't need the score, and lets score-based analyses simply exclude
those clients. Note: the ~1,000 scores *outside* the conventional 300–850 range were also left untouched
for now — the bank's scale may legitimately differ; this is documented as an open item for EDA.

In [5]:
products = ['Current_Account', 'Savings_Account', 'Credit_Card', 'Personal_Loan', 'Home_Loan', 'Vehicle_Loan']

mismatch = pd.Series(False, index=df.index)
for p in products:
    bal = p + '_Balance'
    ghost_balance   = (df[p] == False) & df[bal].notna()   # doesn't hold the product, yet a balance exists
    missing_balance = (df[p] == True)  & df[bal].isna()    # holds the product, but no balance recorded
    print(f'{p:16s}  no-flag-but-balance: {ghost_balance.sum():3d}   flag-but-no-balance: {missing_balance.sum():3d}')
    mismatch |= ghost_balance | missing_balance

df['Flag_Product_Mismatch'] = mismatch
print('\nTotal rows flagged with Flag_Product_Mismatch:', mismatch.sum())

Current_Account   no-flag-but-balance:  38   flag-but-no-balance:  59
Savings_Account   no-flag-but-balance:  45   flag-but-no-balance:  52
Credit_Card       no-flag-but-balance:  55   flag-but-no-balance:  42
Personal_Loan     no-flag-but-balance:  65   flag-but-no-balance:  33
Home_Loan         no-flag-but-balance:  75   flag-but-no-balance:  22
Vehicle_Loan      no-flag-but-balance:  71   flag-but-no-balance:  28

Total rows flagged with Flag_Product_Mismatch: 569


**What:** Created a new boolean column **`Flag_Product_Mismatch`** marking every row where a product
flag contradicts its balance column (balance present while flag = False, or flag = True with no balance).
No values were altered or deleted.

**Why:** We cannot tell *which side* of the contradiction is wrong — the ownership flag or the balance —
so "correcting" either would be an arbitrary guess. Flagging preserves all the information: these rows stay
available for analysis, the flag lets us exclude or inspect them when a specific analysis is sensitive to
the contradiction, and it quantifies the problem for the report.

In [6]:
filled_balances = 0
for p in products:
    bal = p + '_Balance'
    structural = (df[p] == False) & df[bal].isna()
    df.loc[structural, bal] = 0.0
    filled_balances += structural.sum()
print('Structural balance blanks set to 0:', filled_balances)

txn_cols = ['Online_Banking_Txns', 'Mobile_App_Txns', 'USSD_Txns']
txn_missing = int(df[txn_cols].isna().sum().sum())
df[txn_cols] = df[txn_cols].fillna(0)
print('Missing digital transaction counts set to 0:', txn_missing)

Structural balance blanks set to 0: 34882
Missing digital transaction counts set to 0: 13151


**What:** Where a client **does not hold a product** (flag = False, no closed date) and the balance is
blank, the blank was replaced with **0**. Same logic for the digital transaction columns: a blank
transaction count (which in this data only occurs for unregistered channels) was set to **0**.

**Why:** These blanks are **structural, not missing data** — the balance is empty *because* there is no
active account, so its true value is zero, not unknown. This includes previously **closed** accounts: a
closed account with no recorded balance is a settled account, i.e. balance 0. Filling these with 0 makes
aggregations (totals, averages, feature engineering below) work correctly without dropping rows.
One deliberate exception: flag = True with a missing balance stays **NaN** — there the balance genuinely is
unknown (and the row already carries `Flag_Product_Mismatch`). The handful of closed accounts that still
show a *positive* balance are dealt with in the next step.

In [7]:
closed_with_balance = pd.Series(False, index=df.index)
for p in products:
    closed_with_balance |= df[p + '_Closed_Date'].notna() & (df[p + '_Balance'] > 0)

df['Flag_Closed_With_Balance'] = closed_with_balance
print('Closed accounts still carrying a positive balance:', closed_with_balance.sum())

Closed accounts still carrying a positive balance: 15


**What:** Created **`Flag_Closed_With_Balance`** for clients where a product has a closed date but a
positive balance is still recorded. Nothing removed or altered.

**Why:** A lingering balance on a closed account is contradictory, but it may carry real signal (e.g. a
written-off debt, or a data-entry lag around closure). Rather than overthink it, we keep the data intact
and flag it, so it can be examined or excluded later. This follows the same principle as the product
mismatch flag: *issues we chose not to "fix" because they may contain usable information*.

In [8]:
channels = ['Online_Banking', 'Mobile_App', 'USSD']

corrected = 0
for ch in channels:
    wrong = (df[ch + '_Registered'] == False) & (df[ch + '_Txns'] > 0)
    df.loc[wrong, ch + '_Registered'] = True
    corrected += wrong.sum()
print('Registration flags corrected False -> True:', corrected)

for ch in channels:
    leftover = ((df[ch + '_Registered'] == False) & (df[ch + '_Txns'] > 0)).sum()
    assert leftover == 0
print('No channel now shows transactions without registration.')

Registration flags corrected False -> True: 127
No channel now shows transactions without registration.


**What:** Where a client had **transactions on a channel but was marked as not registered**, the
`_Registered` flag was corrected to **True** (130 rows across the three channels). Clients with no
transactions keep their original False flag.

**Why:** Between the two contradicting fields, the transactions are the more trustworthy evidence — you
cannot transact on a channel you don't have, but a registration flag can easily be stale or mis-recorded.
So unlike the balance mismatches (where neither side was clearly right), here the correction direction is
defensible: positive activity implies registration.

In [9]:
closed_cols = [p + '_Closed_Date' for p in products]
df[closed_cols] = df[closed_cols].apply(pd.to_datetime)
print(df[closed_cols].dtypes)

Current_Account_Closed_Date    datetime64[ns]
Savings_Account_Closed_Date    datetime64[ns]
Credit_Card_Closed_Date        datetime64[ns]
Personal_Loan_Closed_Date      datetime64[ns]
Home_Loan_Closed_Date          datetime64[ns]
Vehicle_Loan_Closed_Date       datetime64[ns]
dtype: object


**What:** Converted all six `*_Closed_Date` columns from text to proper **datetime** type.
(All dates parsed cleanly; the audit confirmed there were no malformed or future dates.)

**Why:** As text, dates can't be compared, sorted, or used in time-based calculations. As datetimes we can
compute things like time-since-closure during EDA. Note: when the cleaned table is saved back to CSV the
dates are stored as standardised ISO `YYYY-MM-DD` strings (CSV has no date type) — re-parse with
`pd.read_csv(..., parse_dates=closed_cols)` when loading.

In [10]:
age_band = pd.cut(df['Age'], bins=[17, 30, 40, 50, 60, 100],
                  labels=['18-30', '31-40', '41-50', '51-60', '60+'])

for col in ['Employment_Type', 'Marital_Status', 'Education']:
    before = df[col].isna().sum()
    df[col] = df.groupby(age_band, observed=True)[col].transform(lambda s: s.fillna(s.mode().iloc[0]))
    df[col] = df[col].fillna(df[col].mode().iloc[0])   # safety net if a whole band were empty
    print(f'{col}: filled {before} missing values (now {df[col].isna().sum()} missing)')

Employment_Type: filled 164 missing values (now 0 missing)
Marital_Status: filled 179 missing values (now 0 missing)
Education: filled 180 missing values (now 0 missing)


**What:** Filled missing `Employment_Type`, `Marital_Status` and `Education` with the **most common
value (mode) within the client's age band** (18–30, 31–40, 41–50, 51–60, 60+).

**Why:** These are categorical, so median doesn't apply — mode is the standard choice. Conditioning on age
band makes the guess smarter than a single global mode: a missing employment type for a 65-year-old is far
more likely "Retired", and a missing marital status for a 22-year-old far more likely "Single", than the
overall mode would suggest. This is the same idea as the income imputation below (impute within a relevant
peer group), kept deliberately simple. For `Employment_Type` specifically this age-conditional mode was
chosen over fancier model-based imputation: only ~160 values are missing (<2%), so a simple, explainable
method is appropriate and anything heavier would add complexity without moving results.

In [11]:
suburb_city    = df.dropna(subset=['Suburb', 'City']).groupby('Suburb')['City'].agg(lambda s: s.mode().iloc[0])
city_province  = df.dropna(subset=['City', 'Province']).groupby('City')['Province'].agg(lambda s: s.mode().iloc[0])
suburb_loctype = df.dropna(subset=['Suburb', 'Location_Type']).groupby('Suburb')['Location_Type'].agg(lambda s: s.mode().iloc[0])

for col, source, mapping in [('City', 'Suburb', suburb_city),
                             ('Province', 'City', city_province),
                             ('Location_Type', 'Suburb', suburb_loctype)]:
    before = df[col].isna().sum()
    df[col] = df[col].fillna(df[source].map(mapping))
    print(f'{col}: {before} missing -> {df[col].isna().sum()} after mapping from {source}')

for col in ['City', 'Province', 'Location_Type']:
    df[col] = df[col].fillna(df[col].mode().iloc[0])
df['Suburb'] = df['Suburb'].fillna('Unknown')
print('\nRemaining missing location values:',
      int(df[['Suburb', 'City', 'Province', 'Location_Type']].isna().sum().sum()))

City: 362 missing -> 176 after mapping from Suburb

Province: 301 missing -> 176 after mapping from City
Location_Type: 176 missing -> 176 after mapping from Suburb

Remaining missing location values: 0


**What:** Rebuilt missing location fields using the **geographic hierarchy in the data itself**:
- built lookup tables from complete rows (Suburb → City, City → Province, Suburb → Location_Type);
- a missing City was filled from the client's Suburb, a missing Province from their City, and a missing
  Location_Type from their Suburb — deterministic fills, not guesses, since each suburb belongs to exactly
  one city and each city to one province;
- only where no mapping source existed was the column's most common value used as a last resort;
- a missing Suburb cannot be inferred downward (a city has many suburbs), so it was set to `'Unknown'`
  rather than fabricating a specific suburb.

**Why:** This recovers real information for most gaps (the printed counts show how many were resolved by
mapping vs. fallback), and it is honest about what can't be known — `'Unknown'` is more truthful than
assigning clients to a suburb they may not live in.

In [12]:
before = df['Annual_Income'].isna().sum()

group_median = df.groupby(['Employment_Type', age_band], observed=True)['Annual_Income'].transform('median')
emp_median   = df.groupby('Employment_Type')['Annual_Income'].transform('median')

df['Annual_Income'] = (df['Annual_Income']
                       .fillna(group_median)          # median of same employment type + age band
                       .fillna(emp_median)            # fallback: median of same employment type
                       .fillna(df['Annual_Income'].median()))  # final fallback: overall median

print(f'Annual_Income: filled {before} missing values (remaining: {df["Annual_Income"].isna().sum()})')

Annual_Income: filled 162 missing values (remaining: 0)


**What:** Imputed the missing `Annual_Income` values with the **median income of clients in the same
Employment_Type and age band** (with fallbacks to the employment-type median, then the overall median, in
the rare case a group had no data).

**Why:** A single global median would be badly wrong for many clients — it would hand a student the same
income as a mid-career professional. Grouping by employment type and age band means each imputed value
comes from that client's actual peer group (e.g. a 20-something unemployed client gets the median of other
20-something unemployed clients). Median (not mean) is used because income is heavily right-skewed, so the
median resists distortion by the few very high earners. This "group-wise median imputation" achieves most
of the benefit of sophisticated multivariate methods (like EM on a multivariate normal) while staying
simple, fast, and fully explainable in a report.

In [13]:
df['Flag_No_Products'] = ~df[products].any(axis=1)
print('Clients holding no products at all (flagged, kept):', int(df['Flag_No_Products'].sum()))

Clients holding no products at all (flagged, kept): 576


**What:** Created **`Flag_No_Products`** for clients whose six product flags are all False. They are
**kept** in the dataset.

**Why:** These may be churned clients, data loss, or prospects on the book — we don't know, and they still
carry usable demographic and digital-behaviour information. Flagging lets any product-based analysis
exclude them in one line, without throwing the rows away.

In [14]:
df['Num_Products']         = df[products].sum(axis=1)
df['Num_Digital_Channels'] = df[[c + '_Registered' for c in channels]].sum(axis=1)
df['Total_Digital_Txns']   = df[[c + '_Txns' for c in channels]].sum(axis=1)
df['Num_Prev_Closures']    = df[closed_cols].notna().sum(axis=1)
df['Digitally_Dormant']    = (df['Num_Digital_Channels'] > 0) & (df['Total_Digital_Txns'] == 0)

summary = df[['Num_Products', 'Num_Digital_Channels', 'Total_Digital_Txns', 'Num_Prev_Closures']].describe()
print(summary)
print('\nDigitally dormant clients (registered but zero transactions):', int(df['Digitally_Dormant'].sum()))

       Num_Products  Num_Digital_Channels  Total_Digital_Txns  \
count   9866.000000           9866.000000         9866.000000   
mean       2.429049              1.666430            7.765761   
std        1.280159              0.836522            7.624139   
min        0.000000              0.000000            0.000000   
25%        2.000000              1.000000            2.000000   
50%        2.000000              2.000000            6.000000   
75%        3.000000              2.000000           11.000000   
max        6.000000              3.000000           85.000000   

       Num_Prev_Closures  
count         9866.00000  
mean             0.11717  
std              0.33853  
min              0.00000  
25%              0.00000  
50%              0.00000  
75%              0.00000  
max              2.00000  



Digitally dormant clients (registered but zero transactions): 541


**What:** Engineered five new features:

| Feature | Definition |
|---|---|
| `Num_Products` | count of the six product flags that are True |
| `Num_Digital_Channels` | count of digital channels registered (online, app, USSD) |
| `Total_Digital_Txns` | total transactions across the three channels |
| `Num_Prev_Closures` | number of products with a closed date |
| `Digitally_Dormant` | True if registered on ≥1 digital channel but made **zero** transactions |

**Why:** These roll the 20+ raw product/channel columns up into interpretable client-level measures of
relationship depth (`Num_Products`), digital adoption vs. actual engagement (`Num_Digital_Channels`,
`Total_Digital_Txns`, `Digitally_Dormant`), and past attrition (`Num_Prev_Closures`) — exactly the kind of
variables EDA and modelling will use. They are computed *after* the cleaning steps above so they benefit
from the corrected registration flags and the structural zero-fills.

In [15]:
df.to_csv('SBSA DS Case Study 2027 copy.csv', index=False)
print('Cleaned dataset saved to \'SBSA DS Case Study 2027 copy.csv\'')
print('Final shape:', df.shape)

print('\nColumns still containing NaN (all deliberate):')
miss = df.isna().sum()
print(miss[miss > 0].to_string())

Cleaned dataset saved to 'SBSA DS Case Study 2027 copy.csv'
Final shape: (9866, 44)

Columns still containing NaN (all deliberate):
Current_Account_Balance          59
Savings_Account_Balance          52
Credit_Card_Balance              42
Personal_Loan_Balance            33
Home_Loan_Balance                22
Vehicle_Loan_Balance             28
Current_Account_Closed_Date    9602
Savings_Account_Closed_Date    9595
Credit_Card_Closed_Date        9631
Personal_Loan_Closed_Date      9690
Home_Loan_Closed_Date          9783
Vehicle_Loan_Closed_Date       9739
Credit_Score                    362


## Final state — cleaning summary

Saved back to **`SBSA DS Case Study 2027 copy.csv`** (the original CSV was never modified).

**What was done, in order:**
1. Removed 27 duplicate `CustomerID` rows (13 exact copies, 14 conflicting) — IDs now unique.
2. Removed all under-18 clients (107 rows, including the 3 negative ages) — out of scope + overwhelmingly implausible records.
3. Left missing `Credit_Score` as NaN — no imputation of a high-stakes attribute.
4. Flagged product-flag/balance contradictions (`Flag_Product_Mismatch`) instead of guessing which side is wrong.
5. Filled *structural* blanks with 0 (balances where no product is held; transaction counts on unused channels).
6. Flagged closed accounts still carrying a balance (`Flag_Closed_With_Balance`) — kept, not "fixed".
7. Corrected `_Registered` flags to True where transactions prove the channel was used.
8. Converted the six closed-date columns to datetime (ISO strings in the saved CSV).
9. Imputed `Employment_Type`, `Marital_Status`, `Education` with the age-band mode.
10. Rebuilt missing City / Province / Location_Type via the Suburb→City→Province hierarchy; unresolvable suburbs set to `'Unknown'`.
11. Imputed missing `Annual_Income` with the median of the client's employment-type × age-band peer group.
12. Flagged clients with no products (`Flag_No_Products`) — kept.
13. Engineered `Num_Products`, `Num_Digital_Channels`, `Total_Digital_Txns`, `Num_Prev_Closures`, `Digitally_Dormant`.

**Remaining NaN values are all deliberate:** missing credit scores (no imputation by design), the ~240
balances where the client holds the product but no balance was recorded (genuinely unknown, and carrying
`Flag_Product_Mismatch`), and the closed-date columns (NaN simply means the product was never closed).

**Open item carried into EDA:** ~1,000 credit scores fall outside the conventional 300–850 range; they were
left untouched pending a decision on whether the bank uses a different scale.

**Verdict:** the dataset is de-duplicated, in-scope, type-correct, contradiction-flagged, and fully
documented — **ready to move on to exploratory data analysis.**